In [25]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o-mini")

In [26]:
class State(TypedDict):
    document: str
    summary: str
    sentiment: str
    key_points: str
    recommendation: str
    final_analysis: str


In [27]:
def get_summary(state: State):
    response = llm.invoke(f"이 문서를 3문장으로 요약해줘: {state['document']}")
    return {
        "summary": response.content,
    }

def get_sentiment(state: State):
    response = llm.invoke(f"이 문서의 감정과 어조를 분석해줘: {state['document']}")
    return {
        "sentiment": response.content,
    }

def get_key_points(state: State):
    response = llm.invoke(f"이 문서의 가장 중요한 핵심 포인트 5가지를 정리해줘: {state['document']}")
    return {
        "key_points": response.content,
    }

def get_recommendation(state: State):
    response = llm.invoke(f"이 문서를 바탕으로 추천할 다음 단계 3가지를 제안해줘: {state['document']}")
    return {
        "recommendation": response.content,
    }

def get_final_analysis(state: State):
    response = llm.invoke(
        f"""
        다음 보고서를 종합적으로 분석해줘.

        문서 분석 보고서
        ========================

        요약:
        {state['summary']}
        
        감정 분석:
        {state['sentiment']}
        
        핵심 포인트:
        {state["key_points"]}
        
        추천 사항:
        {state.get('recommendation', "N/A")}
        """)
    return {
        "final_analysis": response.content,
    }

In [28]:
graph_builder = StateGraph(State)

graph_builder.add_node("get_summary", get_summary)
graph_builder.add_node("get_sentiment", get_sentiment)
graph_builder.add_node("get_key_points", get_key_points)
graph_builder.add_node("get_recommendation", get_recommendation)
graph_builder.add_node("get_final_analysis", get_final_analysis)

graph_builder.add_edge(START, "get_summary")
graph_builder.add_edge(START, "get_sentiment")
graph_builder.add_edge(START, "get_key_points")
graph_builder.add_edge(START, "get_recommendation")

graph_builder.add_edge("get_summary", "get_final_analysis")
graph_builder.add_edge("get_sentiment", "get_final_analysis")
graph_builder.add_edge("get_key_points", "get_final_analysis")
graph_builder.add_edge("get_recommendation", "get_final_analysis")
graph_builder.add_edge("get_final_analysis", END)

graph = graph_builder.compile()

#graph

In [29]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()

for chunk in graph.stream(
    {"document": document},
    stream_mode="updates"
):
    print(chunk, "\n")


{'get_summary': {'summary': '연방공개시장위원회(FOMC)는 고용 증가 둔화와 인플레이션 재상승에 대응하기 위해 기준금리를 0.25%포인트 인하하기로 결정했습니다. 현재 실업률과 경제 성장률은 여전히 낮지만, 노동시장과 소비지출 둔화의 신호가 나타나고 있으며, 인플레이션은 목표인 2%보다 높은 수준에 있습니다. 연준은 최대고용과 물가안정을 위한 통화정책을 지속적으로 조정하겠다고 밝혔으며, 앞으로도 변동성과 예상되는 위험에 따라 적절한 대응을 할 것임을 강조했습니다.'}} 

{'get_recommendation': {'recommendation': '이 문서에서 다루어진 주요 주제는 미국의 통화정책, 고용 시장, 인플레이션, 그리고 연방준비제도(Fed)의 최근 결정 내용 등입니다. 이를 바탕으로 다음과 같은 3가지 추천 단계를 제안합니다:\n\n1. **경제 데이터 모니터링 및 분석 강화**:\n   - 고용 시장과 인플레이션 관련 데이터를 지속적으로 모니터링하고, 이를 정기적으로 분석하여 경제의 동향을 정확하게 파악합니다. 데이터 기반의 접근 방식은 통화 정책 결정에 있어 더욱 효과적인 방향성을 제공할 수 있습니다.\n\n2. **정책 소통 및 투명성 제고**:\n   - 연준의 결정 과정 및 정책 목표에 대한 소통을 강화하여 국민 및 시장의 이해를 돕고, 정책의 투명성을 확보합니다. 이러한 소통은 대중의 신뢰를 높이고, 정책이 실제 경제에 미치는 영향을 효과적으로 전달하는 데 기여할 수 있습니다.\n\n3. **위험 관리 프로세스 강화**:\n   - 고용 시장의 하방 위험과 인플레이션의 상방 위험을 고려하여 리스크 관리 프로세스를 강화합니다. 정책 결정 시 발생할 수 있는 다양한 시나리오를 미리 고려하고, 유연한 대응 계획을 마련하여 변화하는 경제 상황에 빠르게 대응할 수 있도록 합니다.\n\n이러한 단계들은 연준의 통화정책 목표인 최대 고용 및 물가 안정을 달성하는 데 기여할 수 있을 것입니다.'}} 

{'get_sentiment